# Evaluating the Big-Five Loop

In this notebook, we are going to evaluate the workflow:

1. Generate a personality description, $p$, based on the specified Big-Five scores, $S$ (with [our utility script](https://github.com/tingsyo/ai-subject/blob/main/persona/utils/generate_persona_from_ocean.py)).
2. Ask LLMs to use $P$ to respond to the **[Big-Five-50](https://ipip.ori.org/new_ipip-50-item-scale.htm)** questionnaire, $Q$ (with [our utility script](https://github.com/tingsyo/ai-subject/blob/main/likert_questionnaire/utils/evaluate_persona.py)).

We want to see if $Q(p)$ can match the original $S$.

Since each $p$ is generated based on the **percentile rank (PR)** of the five personality traits, we need to generate a certain number of $p$s so that we can derive the PR of $Q(p)$.

## Generate Personality Descriptions

We generate the personality descriptions with specified PR ranks of the Big-Five personality traits:

$p = LLM(S)$
$S = (o, c, e, a, n)$

where $o, c, e, a, n \in [0, 100]$.

We don't know how sensitive the $LLM$ is to changes in $S$, so it is hard to determine which $S$ values to use. Hence, we perform a simple estimation before we define the values of $S$. Since the values in $S$ are percentile ranks, an arithmetic sequence from 5 to 95 with an increment of 5 should be a suitable choice. Thus, each of the $(o, c, e, a, n)$ has 10 values, and together they form a set of $10^5 = 100,000$ values.

Although generating $100,000$ personality descriptions is computationally feasible, it might be too costly for a pilot study. While we still want to assess an $ LLM$'s sensitivity in generating personality descriptions, we can fix the PR values of the other four personality traits while varying one. We varied one dimension with 21 values, while the other four dimensions are fixed to `[0, 25, 50, 75, 100]`. Here we have $21 \times 5 \times 5  - 21 = 504$ values of $S_{high}$.

In addition, we add a grid of a lower resolution: `[5, 35, 65, 95]` (an arithmetic sequence from 5 to 95 with an increment of 30) as a supplement to the whole search space of the $(o, c, e, a, n)$. As a result, we have $4^5 = 1,024$ values of $S_{low}$ to capture the larger change.

By combining $S_{high}$ and $S_{low}$ and removing redundant values, we obtain a set of $S$ with $1,529$ values. The generated personality descriptions of  $S_{high}$ and $S_{low}$ can be found in the repo ([here](https://github.com/tingsyo/ai-subject/tree/main/persona/data/bfint5_gemma4e4b_t00) and [here](https://github.com/tingsyo/ai-subject/tree/main/persona/data/gemma4_e4b_00)).

## Respond to the Questionnaire with Generated Personality Description

As shown in the previous notebooks ([01](https://github.com/tingsyo/ai-subject/blob/main/likert_questionnaire/reports/01_Reproducibility_of_LLM_Respondents.ipynb) and [02](https://github.com/tingsyo/ai-subject/blob/main/likert_questionnaire/reports/02_Reproducibility_with_random_seeds.ipynb)), we can reproduce the same response by setting the parameter `temperature=0.0`. We then use the identical LLM calls to get the responses of the questionnaire while varying the personality description.

The responses of $1,529$ personality descriptions can be found [here](https://github.com/tingsyo/ai-subject/tree/main/likert_questionnaire/output/temp00).


## Evaluate the Responses

We obtained the questionnaire from the [Open-Source Psychometrics Project](https://openpsychometrics.org/tests/IPIP-BFFM/). These five scales were developed to measure the Big-Five factor markers reported in the following article: *Goldberg, L. R. (1992). The development of markers for the Big-Five factor structure. Psychological Assessment, 4, 26-42*. The five factors are:

1. Extraversion (E) measures boldness, energy, and social interactivity.
2. Agreeableness (A) measures kindness, helpfulness, and willingness to cooperate.
3. Conscientiousness (C) measures self-control, diligence, and attention to detail.
4. Emotional Stability (negative neuroticism, -N) measures depression, irritability, and proneness to anxiety.
5. Intellect/Imagination (openness, O) measures creativity, curiosity, and willingness to entertain new ideas.

For an easier scoring process, we use the following scheme:

```
[
{ "id": 1, "text": "Am the life of the party", "domain": "E", "keyed": "+" },
{ "id": 2, "text": "Feel little concern for others", "domain": "A", "keyed": "-" },
{ "id": 3, "text": "Am always prepared", "domain": "C", "keyed": "+" },
{ "id": 4, "text": "Get stressed out easily", "domain": "N", "keyed": "+" },
{ "id": 5, "text": "Have a rich vocabulary", "domain": "O", "keyed": "+" },
{ "id": 6, "text": "Don't talk a lot", "domain": "E", "keyed": "-" },
{ "id": 7, "text": "Am interested in people", "domain": "A", "keyed": "+" },
{ "id": 8, "text": "Leave my belongings around", "domain": "C", "keyed": "-" },
{ "id": 9, "text": "Am relaxed most of the time", "domain": "N", "keyed": "-" },
{ "id": 10, "text": "Have difficulty understanding abstract ideas", "domain": "O", "keyed": "-" },
{ "id": 11, "text": "Feel comfortable around people", "domain": "E", "keyed": "+" },
{ "id": 12, "text": "Insult people", "domain": "A", "keyed": "-" },
{ "id": 13, "text": "Pay attention to details", "domain": "C", "keyed": "+" },
{ "id": 14, "text": "Worry about things", "domain": "N", "keyed": "+" },
{ "id": 15, "text": "Have a vivid imagination", "domain": "O", "keyed": "+" },
{ "id": 16, "text": "Keep in the background", "domain": "E", "keyed": "-" },
{ "id": 17, "text": "Sympathize with others' feelings", "domain": "A", "keyed": "+" },
{ "id": 18, "text": "Make a mess of things", "domain": "C", "keyed": "-" },
{ "id": 19, "text": "Seldom feel blue", "domain": "N", "keyed": "-" },
{ "id": 20, "text": "Am not interested in abstract ideas", "domain": "O", "keyed": "-" },
{ "id": 21, "text": "Start conversations", "domain": "E", "keyed": "+" },
{ "id": 22, "text": "Am not interested in other people's problems", "domain": "A", "keyed": "-" },
{ "id": 23, "text": "Get chores done right away", "domain": "C", "keyed": "+" },
{ "id": 24, "text": "Am easily disturbed", "domain": "N", "keyed": "+" },
{ "id": 25, "text": "Have excellent ideas", "domain": "O", "keyed": "+" },
{ "id": 26, "text": "Have little to say", "domain": "E", "keyed": "-" },
{ "id": 27, "text": "Have a soft heart", "domain": "A", "keyed": "+" },
{ "id": 28, "text": "Often forget to put things back in their proper place", "domain": "C", "keyed": "-" },
{ "id": 29, "text": "Get upset easily", "domain": "N", "keyed": "+" },
{ "id": 30, "text": "Do not have a good imagination", "domain": "O", "keyed": "-" },
{ "id": 31, "text": "Talk to a lot of different people at parties", "domain": "E", "keyed": "+" },
{ "id": 32, "text": "Am not really interested in others", "domain": "A", "keyed": "-" },
{ "id": 33, "text": "Like order", "domain": "C", "keyed": "+" },
{ "id": 34, "text": "Change my mood a lot", "domain": "N", "keyed": "+" },
{ "id": 35, "text": "Am quick to understand things", "domain": "O", "keyed": "+" },
{ "id": 36, "text": "Don't like to draw attention to myself", "domain": "E", "keyed": "-" },
{ "id": 37, "text": "Take time out for others", "domain": "A", "keyed": "+" },
{ "id": 38, "text": "Shirk my duties", "domain": "C", "keyed": "-" },
{ "id": 39, "text": "Have frequent mood swings", "domain": "N", "keyed": "+" },
{ "id": 40, "text": "Use difficult words", "domain": "O", "keyed": "+" },
{ "id": 41, "text": "Don't mind being the center of attention", "domain": "E", "keyed": "+" },
{ "id": 42, "text": "Feel others' emotions", "domain": "A", "keyed": "+" },
{ "id": 43, "text": "Follow a schedule", "domain": "C", "keyed": "+" },
{ "id": 44, "text": "Get irritated easily", "domain": "N", "keyed": "+" },
{ "id": 45, "text": "Spend time reflecting on things", "domain": "O", "keyed": "+" },
{ "id": 46, "text": "Am quiet around strangers", "domain": "E", "keyed": "-" },
{ "id": 47, "text": "Make people feel at ease", "domain": "A", "keyed": "+" },
{ "id": 48, "text": "Am exacting in my work", "domain": "C", "keyed": "+" },
{ "id": 49, "text": "Often feel blue", "domain": "N", "keyed": "+" },
{ "id": 50, "text": "Am full of ideas", "domain": "O", "keyed": "+" },
];
```

### Read and Parse Response Files


In [2]:
import io, os, json
import pandas as pd

DATA_DIR = '../output/temp00/'

response_files = os.listdir(DATA_DIR)
print("Total files: "+str(len(response_files)))
print(response_files[100])

Total files: 1529
temp00-bfpr-005-005-005-095-065.csv


In [4]:
def parse_file_name(file: str):
    ''' Convert the file name back to the OCEAN scores '''
    substr = file.replace(".csv","").split("-")
    scores = [int(s) for s in substr[-5:]]
    return(scores)

parse_file_name(response_files[100])

[5, 5, 5, 95, 65]

In [7]:
# Scoring

def score_big_five_50(responses):
    ''' Score the responses of the Big Five %0 questionnaire. '''
    scoring_scheme = {
        "E": { "plus": [1, 11, 21, 31, 41], "minus": [6, 16, 26, 36, 46] },
        "A": { "plus": [7, 17, 27, 37, 42, 47], "minus": [2, 12, 22, 32] },
        "C": { "plus": [3, 13, 23, 33, 43, 48], "minus": [8, 18, 28, 38] },
        "N": { "plus": [4, 14, 24, 29, 34, 39, 44, 49], "minus": [9, 19] },
        "O": { "plus": [5, 15, 25, 35, 40, 45, 50], "minus": [10, 20, 30] },
    }
    scores = {}
    for trait in scoring_scheme.keys():
        score = 0
        # Plus items
        for item in scoring_scheme[trait]["plus"]:
            score += responses[item-1]
        # Minus items
        for item in scoring_scheme[trait]["minus"]:
            score += (6-responses[item-1])
        # Assign score
        scores[trait] = score
    return(scores)

df = pd.read_csv(DATA_DIR+response_files[100])
print(parse_file_name(response_files[100]))
print(score_big_five_50(df["rating"]))

[5, 5, 5, 95, 65]
{'E': 19, 'A': 49, 'C': 39, 'N': 37, 'O': 38}
